# Notebook 3: Bringing It Together — Visualizing Next-Word Prediction

**Goal:** Watch a real, trained transformer do the thing: predict the next word, live. This ties together attention (Notebook 1) and embeddings/tokenization (Notebook 2) into one running model.

**Time box: ~20-25 minutes.**


## Setup

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

print("Loaded:", MODEL_NAME)


## 1. Next-token prediction, visualized

Feed the model a sentence and look at what it thinks comes next -- not just its single best guess, but the full probability distribution over the top candidates.

In [ ]:
def show_next_word_predictions(text, top_k=10):
    input_ids = tokenizer.encode(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits[0, -1, :]  # logits for the next token
        probs = F.softmax(logits, dim=-1)

    top_probs, top_indices = torch.topk(probs, top_k)
    top_tokens = [tokenizer.decode([idx]) for idx in top_indices]

    plt.figure(figsize=(8, 4))
    plt.barh(range(top_k), top_probs.detach().numpy()[::-1])
    plt.yticks(range(top_k), [repr(t) for t in top_tokens][::-1])
    plt.xlabel("Probability")
    plt.title(f'Next-word predictions after: "{text}"')
    plt.tight_layout()
    plt.show()

show_next_word_predictions("The capital of France is")


**Try it yourself:** edit the sentence below (or the cell above) and re-run. Some ideas:
- `"I opened the fridge and took out a"`
- `"Once upon a"`
- An incomplete idiom: `"The early bird catches the"`
- A sentence with an ambiguous continuation and see how spread out the probabilities are

In [ ]:
show_next_word_predictions("I opened the fridge and took out a")


## 2. From prediction to generation

Next-word prediction, repeated: predict the next token, append it, predict again. This is greedy decoding -- always picking the single most likely next token. Watch the sentence grow one word at a time.

In [ ]:
def generate_greedy(text, num_steps=15):
    input_ids = tokenizer.encode(text, return_tensors="pt")
    for step in range(num_steps):
        with torch.no_grad():
            outputs = model(input_ids)
            next_token_logits = outputs.logits[0, -1, :]
            next_token_id = torch.argmax(next_token_logits).unsqueeze(0).unsqueeze(0)
        input_ids = torch.cat([input_ids, next_token_id], dim=1)
        print(tokenizer.decode(input_ids[0]))
    return tokenizer.decode(input_ids[0])

final_text = generate_greedy("The best way to learn about transformers is")


**Try it yourself:** change the starting sentence and/or `num_steps` above. Notice how greedy decoding can get repetitive -- why might always picking the single most likely word lead to that?

## 3. (If time allows) Real attention, on a real trained model

Back in Notebook 1 we looked at attention weights on random, untrained embeddings. Let's look at the same kind of heatmap, but now on this trained model with a real sentence.

In [ ]:
def plot_model_attention(text, layer=0):
    input_ids = tokenizer.encode(text, return_tensors="pt")
    tokens = [tokenizer.decode([t]) for t in input_ids[0]]

    with torch.no_grad():
        outputs = model(input_ids, output_attentions=True)
    # outputs.attentions: tuple of (num_layers) tensors, each (batch, num_heads, seq_len, seq_len)
    attn = outputs.attentions[layer][0]  # (num_heads, seq_len, seq_len)

    num_heads = attn.shape[0]
    fig, axes = plt.subplots(1, min(num_heads, 4), figsize=(4 * min(num_heads, 4), 4))
    if min(num_heads, 4) == 1:
        axes = [axes]
    for h in range(min(num_heads, 4)):
        ax = axes[h]
        ax.imshow(attn[h].numpy(), cmap="viridis")
        ax.set_xticks(range(len(tokens)))
        ax.set_yticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha="right")
        ax.set_yticklabels(tokens)
        ax.set_title(f"Layer {layer}, Head {h}")
    plt.tight_layout()
    plt.show()

plot_model_attention("The trophy didn't fit in the suitcase because it was too big", layer=0)


**Try it yourself:** change `layer` (try layer 0 vs. a later layer -- check `model.config.n_layer` for the max) and see if attention patterns look more "local" in early layers vs. more spread out in later layers.

### Closing discussion

- Compare this heatmap to the ones from Notebook 1. What's different about attention on a *trained* model with a *real* sentence, versus random untrained embeddings?
- In the ambiguous sentence above ("it" could refer to the trophy or the suitcase), does any head seem to resolve the ambiguity correctly?
- We went from raw attention math (Notebook 1) → the vectors that feed it (Notebook 2) → a trained model doing real prediction and generation (Notebook 3). Which part surprised you most?
